In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import time
from datetime import datetime, timedelta
from collections import deque


HISTORY_FILE = "planet_history.csv"

def load_history():
    if os.path.exists(HISTORY_FILE):
        return pd.read_csv(HISTORY_FILE, parse_dates=["timestamp"])
    return pd.DataFrame()

def save_history(df):
    df.to_csv(HISTORY_FILE, index=False)





def run_hourly_pipeline():    
    #below we need to run this hourly
    BASE_URL = "https://api.helldivers2.dev/api"
    #try and get all planets
    planets_url = "/v1/planets"
    import requests
    import time
    import pandas as pd

    #need to add headers due to api rules for pulling
    headers = {
        "X-Super-Client": "helldivers-machine_learning-project",
        "X-Super-Contact": "jacksonwaiver@gmail.com"
    }
    url = f"{BASE_URL}/v1/planets"
    response = requests.get(url, headers = headers)

    #print("Status:", response.status_code)
    #print("Content-Type:", response.headers.get("Content-Type"))
    #print(response.text[:200])  # debug peek

    planets = response.json()

    #print(len(planets))


    #print(planets[0])



    url_major_order = "https://api.helldivers2.dev/raw/api/v2/Assignment/War/801"

    #need to add headers due to api rules for pulling
    headers = {
        "X-Super-Client": "helldivers-machine_learning-project",
        "X-Super-Contact": "jacksonwaiver@gmail.com"
    }

    response = requests.get(url_major_order, headers = headers)

    #print("Status:", response.status_code)
    #print("Content-Type:", response.headers.get("Content-Type"))
    #print(response.text[:200])  # debug peek

    major_order_planets = response.json()
    if len(major_order_planets) == 0:
        major_order_length = 0
        #in_major_order = "F"
    else: 
        major_order_length = len(major_order_planets[0]['setting']['tasks'])


    major_order_planet_list_index = []
    for i in range(0, major_order_length, 1):
        #get planet index in current major order
        planet_index = major_order_planets[0]['setting']['tasks'][i]['values'][2]
        major_order_planet_list_index.append(planet_index)
        
    print(major_order_planets)

    major_order_dispatch_url = "https://api.helldivers2.dev/api/v1/assignments"

    response = requests.get(major_order_dispatch_url, headers = headers)

    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print(response.text[:200])  # debug peek

    major_order_dispatch = response.json()



    planets_names_major_order = []


    #try and get all planets indices
    planets_url = "/v1/planets"
    import requests

    #need to add headers due to api rules for pulling
    headers = {
        "X-Super-Client": "helldivers-machine_learning-project",
        "X-Super-Contact": "jacksonwaiver@gmail.com"
    }
    # time.sleep(10)
    # #will bomb out with error 429 due to github api pull restrictions if 5 GET requests within 10 seconds, so count
    # count = 0
    # for i in major_order_planet_list_index:
    #     #wait 10 seconds
    #     if count % 5 == 0:
    #         print("I AM STUCK")
    #         time.sleep(10)
    #         print("I AM NOT STUCK")
    #     url = f"{BASE_URL}/v1/planets/" + str(i)
    #     response = requests.get(url, headers = headers)
    #     count += 1
    #     #increase count
        
    #     print("Status:", response.status_code)
    #     print("Content-Type:", response.headers.get("Content-Type"))
    #     print(response.text[:200])  # debug peek

    #     planets_grabbed = response.json()
    #     #now append grabbed planets to list
    #     planets_names_major_order.append(planets_grabbed)
    #     if count == 50:
    #         #reset, so computation aren't so hard
    #         count = 0


    #     #could definitely go through the list and find the planets with those indices

    for i in major_order_planet_list_index:
        planets_names_major_order.append(planets[i])
        


    dss_url = "https://api.helldivers2.dev/api/v2/space-stations"

    response = requests.get(dss_url, headers = headers)

    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print(response.text[:200])  # debug peek

    dss = response.json()
    #df['Y-M-D'] = datetime.now().strftime("%Y-%m-%d")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    #print(now)

    #df['CST_H_M_S'] = now = datetime.now().strftime("%H:%M:%S")
    dss_planet_orbited = dss[0]['planet']['name']

    planet_rows_list = []
    in_major_order = "F"
    for i in planets:

        if i['index'] in major_order_planet_list_index:
            in_major_order = "T"

        #put out the faction names for easier use later

        if i['currentOwner'] == "Humans":
            faction = "Humans"
        elif i['currentOwner'] == "Terminids":
            faction = "Terminids"
        elif i['currentOwner'] == "Illuminate":
            faction = "Illuminate"
        elif i['currentOwner'] == "Automaton":
            faction = "Automaton"

        #make df
        planet_rows = {
            "planet_index": i['index'],
            "name": i['name'],
            "sector": i['sector'],
            "biome": i['biome']['name'],
            "planet_environ_hazards": i['hazards'][0]['name'],
            "player_on_planet": i['statistics']['playerCount'],
            "isAvailableToPlay": "F",
            "in_major_order": in_major_order,
            "missions_won": i['statistics']['missionsWon'],
            "missions_lost": i['statistics']['missionsLost'],
            "is_humans_defending": "F",
            "helldiver_deaths_total": i['statistics']['deaths'],
            #below is a place holder for now, will calculate the helldiver diff total later
            "helldiver_deaths_1hour_diff": "0",
            "bullets_fired": i['statistics']['bulletsFired'],
            "bullets_hit": i['statistics']['bulletsHit'],
            "illuminate_killed": i['statistics']['illuminateKills'],
            "terminids_killed": i['statistics']['terminidKills'],
            "automatons_killed": i['statistics']['automatonKills'],
            "currentOwner": i['currentOwner'],
            "major_order_completed": "F",
            "planet_health": i['health'],
            "max_planet_health": i['maxHealth']
        }
        planet_rows_list.append(planet_rows)
        
        #set to default 
        in_major_order = "F"

    df = pd.DataFrame(planet_rows_list)
    if major_order_length == 0:
        df['major_order_dispatch'] = "NONE"
    else: 
        df['major_order_dispatch'] = major_order_dispatch[0]['briefing']



    import networkx as nx

    G = nx.Graph()


    def add_edge_by_name(G, name_a, name_b, name_to_index):
        a = name_to_index[name_a]
        b = name_to_index[name_b]
        G.add_edge(a, b)


    name_to_index = {
        row["name"]: row["planet_index"]
        for _, row in df.iterrows()
    }


    # 1. Build all nodes from your DataFrame
    for _, row in df.iterrows():
        G.add_node(
            row["planet_index"],
            name=row["name"],
            owner=row["currentOwner"]
        )

    owner_by_index = {
        row["planet_index"]: row["currentOwner"]
        for _, row in df.iterrows()
    }


    def shortest_path_with_major_order_rules(G, owner_by_index, major_order_planets, target):
        major_order_planets = set(major_order_planets)

        # Your frontline rule, reused
        def is_frontline(planet):
            if owner_by_index.get(planet) == "Humans":
                return False
            for n in G.neighbors(planet):
                if owner_by_index.get(n) == "Humans":
                    return True
            return False

        # All current human planets are valid starting points
        start_planets = [p for p, o in owner_by_index.items() if o == "Humans"]
        if not start_planets:
            return None

        queue = deque()
        visited = set(start_planets)

        for h in start_planets:
            queue.append((h, [h]))

        while queue:
            current, path = queue.popleft()

            for neighbor in G.neighbors(current):
                if neighbor in visited:
                    continue

                allowed = False

                # Rule 1: Humans always allowed
                if owner_by_index.get(neighbor) == "Humans":
                    allowed = True

                # Rule 2: Major order planets always allowed
                elif neighbor in major_order_planets:
                    allowed = True

                # Rule 3: Non-major planets must be frontline playable
                elif is_frontline(neighbor):
                    allowed = True

                if not allowed:
                    continue

                visited.add(neighbor)
                new_path = path + [neighbor]

                # Stop as soon as we reach the target
                if neighbor == target:
                    return new_path

                queue.append((neighbor, new_path))

        return None



    def recompute_major_order_flags(df, G, owner_by_index, major_order_planets):
        major_order_planets = set(major_order_planets)

        # Reset dynamic flags
        df["in_major_order"] = "F"
        df["major_order_completed"] = "F"

        # Mark completed Major Order planets (only true targets, only if Humans own them)
        for planet in major_order_planets:
            if owner_by_index.get(planet) == "Humans":
                df.loc[df["planet_index"] == planet, "major_order_completed"] = "T"

        # For each ACTIVE Major Order target, find its path and mark it
        for target in major_order_planets:
            # If already completed, skip path marking
            if owner_by_index.get(target) == "Humans":
                continue

            path = shortest_path_with_major_order_rules(
                G, owner_by_index, major_order_planets, target
            )

            if path is None:
                continue

            # path = [human_start, ..., target]
            # Everything except the human start is currently part of the Major Order corridor
            for p in path[1:]:
                df.loc[df["planet_index"] == p, "in_major_order"] = "T"


    def get_enemy_attacking_planet_and_owner(G, planet_index, owner_by_index):
        my_owner = owner_by_index.get(planet_index)

        # Only meaningful when Humans own the planet (defense)
        if my_owner != "Humans":
            return None, None

        for neighbor in G.neighbors(planet_index):
            neighbor_owner = owner_by_index.get(neighbor)

            # Enemy attacker = not Humans
            if neighbor_owner != "Humans":
                return neighbor, neighbor_owner

        return None, None


    #already made edge from ESTANU TO CRIMSICA
    #DON'T CONNECT ABOVE PHACT BAY UP OR UPPER RIGHT OR UPPER LEFT WHEN ITS MENTIONED IN COMMENTS, WHEN MENTIONED IN COMMENT THAT IS CALLED THE HUB




    #WE ARE IN TERMINID TERRITORY ON MIDDLE RIGHT



    #now make rest of edges
    add_edge_by_name(G, "ESTANU", "CRIMSICA", name_to_index)

    #CONNECT EVERYTHING TO CRUCIBLE
    add_edge_by_name(G, "CRUCIBLE", "VOLTERRA", name_to_index)
    add_edge_by_name(G, "CRUCIBLE", "KRAKATWO", name_to_index)

    #CONNECT EVERYTHING TO KRAKATWO
    add_edge_by_name(G, "KRAKATWO", "SLIF", name_to_index)
    add_edge_by_name(G, "KRAKATWO", "NUBLARIA I", name_to_index)

    #CONNECT EVERYTHING TO SLIF
    add_edge_by_name(G,"SLIF", "VELD", name_to_index)

    #CONNECT EVERYTHING TO NUBLARIA I
    add_edge_by_name(G, "NUBLARIA I","SULFURA", name_to_index)
    add_edge_by_name(G, "NUBLARIA I","FORT UNION", name_to_index)

    #CONNECT EVERYTHING TO SULFURA
    add_edge_by_name(G,"SULFURA", "AZTERRA", name_to_index)

    #CONNECT EVERYTHING TO AZTERRA
    add_edge_by_name(G, "AZTERRA", "TERREK", name_to_index)
    add_edge_by_name(G, "AZTERRA", "CIRRUS", name_to_index)

    #CONNECT EVERYTHING TO FORT UNION
    add_edge_by_name(G, "FORT UNION", "CIRRUS", name_to_index)

    #CONNECT EVERYTHING TO CIRRUS
    add_edge_by_name(G, "CIRRUS", "TERREK", name_to_index)
    add_edge_by_name(G, "CIRRUS", "HEETH", name_to_index)

    #CONNECT EVERYTHING TO TERREK
    add_edge_by_name(G,"TERREK", "BORE ROCK", name_to_index)

    #CONNECT EVERYTHING TO VOLTERRA
    add_edge_by_name(G, "VOLTERRA", "CARAMOOR", name_to_index)
    add_edge_by_name(G, "VOLTERRA", "SLIF", name_to_index)
    add_edge_by_name(G, "VOLTERRA", "ALTA V", name_to_index)

    #CONNECT EVERYTHING TO HEETH
    add_edge_by_name(G, "HEETH", "FENRIR III", name_to_index)
    add_edge_by_name(G, "HEETH", "ERATA PRIME", name_to_index)

    #CONNECT EVERYTHING TO ERATA PRIME
    add_edge_by_name(G, "ERATA PRIME", "BORE ROCK", name_to_index)
    add_edge_by_name(G, "ERATA PRIME", "NIVEL 43", name_to_index)

    #CONNECT EVERYTHING TO ALTA V
    add_edge_by_name(G, "ALTA V", "INARI", name_to_index)
    add_edge_by_name(G, "ALTA V", "CARAMOOR", name_to_index)
    add_edge_by_name(G, "ALTA V", "SLIF", name_to_index)

    #CONNECT EVERYTHING TO INARI
    add_edge_by_name(G, "INARI", "VELD", name_to_index)
    add_edge_by_name(G, "INARI", "URSICA XI", name_to_index)

    #CONNECT EVERYTHING TO URSICA XI
    add_edge_by_name(G, "URSICA XI", "ACHIRD III", name_to_index)
    add_edge_by_name(G, "URSICA XI", "ACHERNAR SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO ACHERNAR SECUNDUS 
    add_edge_by_name(G, "ACHERNAR SECUNDUS", "GAR HAREN", name_to_index)
    add_edge_by_name(G,"ACHERNAR SECUNDUS", "DARIUS II", name_to_index)

    #CONNECT EVERYTHING TO GAR HAREN
    add_edge_by_name(G, "GAR HAREN", "GRAND ERRANT", name_to_index)
    add_edge_by_name(G, "GAR HAREN", "GATRIA", name_to_index)
    add_edge_by_name(G, "GAR HAREN", "PHACT BAY", name_to_index)

    #CONNECT EVERYTHING TO GRAND ERRANT
    add_edge_by_name(G, "GRAND ERRANT", "POLARIS PRIME", name_to_index)
    add_edge_by_name(G, "GRAND ERRANT", "PHERKAD SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO POLARIS PRIME
    add_edge_by_name(G,"POLARIS PRIME", "PHERKAD SECUNDUS", name_to_index)
    add_edge_by_name(G,"POLARIS PRIME", "POLLUX 31", name_to_index)

    #CONNECT EVERYTHING TO POLLUX 31
    add_edge_by_name(G, "POLLUX 31", "PRASA", name_to_index)

    #CONNECT EVERYTHING TO GATRIA 
    add_edge_by_name(G, "GATRIA", "TRANDOR", name_to_index)
    add_edge_by_name(G, "GATRIA", "PHACT BAY", name_to_index)
    add_edge_by_name(G, "GATRIA", "PHERKAD SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO TRANDOR
    add_edge_by_name(G, "TRANDOR", "PEACOCK", name_to_index)

    #CONNECT EVERYTHING TO PEACOCK
    add_edge_by_name(G, "PEACOCK", "PARTION", name_to_index)

    #CONNECT EVERYTHING TO PHACT BAY
    add_edge_by_name(G, "PHACT BAY", "GATRIA", name_to_index)
    add_edge_by_name(G, "PHACT BAY", "GAR HAREN", name_to_index)
    add_edge_by_name(G, "PHACT BAY", "DARIUS II", name_to_index)
    add_edge_by_name(G, "PHACT BAY", "PANDION-XXIV", name_to_index)
    add_edge_by_name(G, "PHACT BAY", "PARTION", name_to_index)

    #CONNECT EVERYTHING TO ACHIRD III
    add_edge_by_name(G, "ACHIRD III","URSICA XI", name_to_index)
    add_edge_by_name(G, "ACHIRD III","DARIUS II", name_to_index)
    #JUST ADDED BELOW
    add_edge_by_name(G, "ACHIRD III","TURING", name_to_index)

    #CONNECT EVERYTHING TO TURING
    add_edge_by_name(G, "TURING", "VELD", name_to_index)
    add_edge_by_name(G, "TURING", "ACAMAR IV", name_to_index)


    #DO ABOVE PANDION AND INCLUDE PANDION

    #CONNECT EVERYTHING TO PANDION-XXIV
    add_edge_by_name(G, "PANDION-XXIV", "PHACT BAY", name_to_index)
    add_edge_by_name(G, "PANDION-XXIV", "GACRUX", name_to_index)
    add_edge_by_name(G, "PANDION-XXIV", "ACAMAR IV", name_to_index)

    #CONNECT EVERYTHING TO PARTION
    add_edge_by_name(G, "PARTION", "PHACT BAY", name_to_index)
    add_edge_by_name(G, "PARTION", "PEACOCK", name_to_index)
    add_edge_by_name(G, "PARTION", "GACRUX", name_to_index)
    add_edge_by_name(G, "PARTION", "OVERGOE PRIME", name_to_index)

    #CONNECT EVERYTHING TO ACAMAR IV
    add_edge_by_name(G, "ACAMAR IV", "DARIUS II", name_to_index)
    add_edge_by_name(G, "ACAMAR IV", "PANDION-XXIV", name_to_index)
    add_edge_by_name(G, "ACAMAR IV", "TURING", name_to_index)
    add_edge_by_name(G, "ACAMAR IV", "CRIMSICA", name_to_index)
    add_edge_by_name(G, "ACAMAR IV", "GACRUX", name_to_index)

    #CONNECT EVERYTHING TO OVERGOE PRIME
    add_edge_by_name(G, "OVERGOE PRIME", "AZUR SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO CRIMSICA
    add_edge_by_name(G, "CRIMSICA", "FORI PRIME", name_to_index)
    add_edge_by_name(G, "CRIMSICA", "ESTANU", name_to_index)

    #CONNECT EVERYTHING TO ESTANU
    add_edge_by_name(G, "ESTANU", "FORI PRIME", name_to_index)
    add_edge_by_name(G, "ESTANU", "HELLMIRE", name_to_index)

    #CONNECT EVERYTHING TO FORI PRIME
    add_edge_by_name(G, "FORI PRIME", "GACRUX", name_to_index)
    add_edge_by_name(G, "FORI PRIME", "OSHAUNE", name_to_index)

    #CONNECT EVERYTHING TO NAVI VII
    add_edge_by_name(G, "NAVI VII", "NABATEA SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO OSHAUNE
    add_edge_by_name(G, "OSHAUNE", "HELLMIRE", name_to_index)
    add_edge_by_name(G, "OSHAUNE", "OMICRON", name_to_index)

    #CONNECT EVERYTHING TO HELLMIRE
    add_edge_by_name(G, "HELLMIRE", "ERATA PRIME", name_to_index)
    add_edge_by_name(G, "HELLMIRE", "FENRIR III", name_to_index)
    add_edge_by_name(G, "HELLMIRE", "NIVEL 43", name_to_index)
    add_edge_by_name(G, "HELLMIRE", "OSHAUNE", name_to_index)

    #CONNECT EVERYTHING TO NIVEL 43
    add_edge_by_name(G, "NIVEL 43", "HELLMIRE", name_to_index)
    add_edge_by_name(G, "NIVEL 43", "ERATA PRIME", name_to_index)
    add_edge_by_name(G, "NIVEL 43", "ZAGON PRIME", name_to_index)
    add_edge_by_name(G, "NIVEL 43", "ERSON SANDS", name_to_index)
    add_edge_by_name(G, "NIVEL 43", "ESKER", name_to_index)

    #CONNECT EVERYTHING TO ZAGON PRIME
    add_edge_by_name(G, "ZAGON PRIME", "OMICRON", name_to_index)


    #CONNECT EVERYTHING TO OMICRON
    add_edge_by_name(G, "OMICRON", "OSHAUNE", name_to_index)

    #CONNECT EVERYTHING TO NABATEA SECUNDUS
    add_edge_by_name(G, "NABATEA SECUNDUS", "GEMSTONE BLUFFS", name_to_index)

    #CONNECT EVERYTHING TO GEMSTONE BLUFFS
    add_edge_by_name(G, "GEMSTONE BLUFFS", "EPSILON PHOENCIS VI", name_to_index)
    add_edge_by_name(G, "GEMSTONE BLUFFS", "DIASPORA X", name_to_index)

    #CONNECT EVERYTHING TO ENULIAE
    add_edge_by_name(G, "ENULIALE", "DIASPORA X", name_to_index)
    add_edge_by_name(G, "ENULIALE", "EPSILON PHOENCIS VI", name_to_index)

    #CONNECT EVERYTHING TO ERSON SANDS
    add_edge_by_name(G, "ERSON SANDS", "ESKER", name_to_index)
    add_edge_by_name(G, "ERSON SANDS", "SOCORRO III", name_to_index)

    #CONNECT EVERYTHING TO ESKER
    add_edge_by_name(G,"ESKER","BORE ROCK", name_to_index)



    #START AT SHALLUS FOR AUTOMATONS AT TOP MIDDLE 


    #CONNECT EVERYTHING TO SHALLUS
    add_edge_by_name(G, "SHALLUS", "SHELT", name_to_index)
    add_edge_by_name(G, "SHALLUS", "MASTIA", name_to_index)

    #CONNECT EVERYTHING TO SHELT
    add_edge_by_name(G, "SHELT", "MASTIA", name_to_index)
    add_edge_by_name(G, "SHELT", "IMBER", name_to_index)

    #CONNECT EVERYTHING TO IMBER
    add_edge_by_name(G, "IMBER", "GAELLIVARE", name_to_index)
    add_edge_by_name(G, "IMBER", "CLAORELL", name_to_index)

    #CONNECT EVERYTHING TO CLAORELL
    add_edge_by_name(G, "CLAORELL", "VOG-SOJOTH", name_to_index)
    add_edge_by_name(G, "CLAORELL", "CLASA", name_to_index)

    #CONNECT EVERYTHING TO CLASA
    add_edge_by_name(G, "CLASA", "DEMIURG", name_to_index)
    add_edge_by_name(G, "CLASA", "ZEFIA", name_to_index)
    add_edge_by_name(G, "CLASA", "YED PRIOR", name_to_index)

    #CONNECT EVERYTHING TO ZEFIA
    add_edge_by_name(G, "ZEFIA", "MINTORIA", name_to_index)
    add_edge_by_name(G, "ZEFIA", "YED PRIOR", name_to_index)

    #CONNECT EVERYTHING TO YED PRIOR
    add_edge_by_name(G, "YED PRIOR", "BLISTICA", name_to_index)

    #CONNECT EVERYTHING TO BLISTICA
    add_edge_by_name(G,"BLISTICA", "MINTORIA", name_to_index)
    add_edge_by_name(G,"BLISTICA", "ZZANIAH PRIME", name_to_index)

    #CONNECT EVERYTHING TO ZZANIAH PRIME
    add_edge_by_name(G, "ZZANIAH PRIME", "ZOSMA", name_to_index)


    #CONNECT EVERYTHING TO MASTIA
    add_edge_by_name(G, "MASTIA", "GAELLIVARE", name_to_index)
    add_edge_by_name(G, "MASTIA", "FENMIRE", name_to_index)
    add_edge_by_name(G, "MASTIA", "TARSH", name_to_index)

    #CONNECT EVERYTHING TO GAELLIVARE
    add_edge_by_name(G, "GAELLIVARE", "LESATH", name_to_index)

    #CONNECT EVERYTHING TO LESATH
    add_edge_by_name(G, "LESATH", "VERNEN WELLS", name_to_index)
    add_edge_by_name(G, "LESATH", "MENKENT", name_to_index)
    add_edge_by_name(G, "LESATH", "CHORT BAY", name_to_index)
    add_edge_by_name(G, "LESATH", "PENTA", name_to_index)
    add_edge_by_name(G, "LESATH", "VOG-SOJOTH", name_to_index)

    #CONNECT EVERYTHING TO PENTA

    #COMMENTED OUT BECAUSE DARK FLUID BEAM VIA DSS DESTROYED PENTA

    # add_edge_by_name(G, "PENTA", "CHORT BAY", name_to_index)
    # add_edge_by_name(G, "PENTA", "MERAK", name_to_index)

    #CONNECT EVERYTHING TO CHORT BAY
    add_edge_by_name(G,"CHORT BAY", "MENKENT",  name_to_index)
    add_edge_by_name(G,"CHORT BAY", "CHOOHE",  name_to_index)

    #CONNECT EVERYTHING TO CHOOHE
    add_edge_by_name(G,"CHOOHE", "MENKENT",  name_to_index)
    add_edge_by_name(G,"CHOOHE", "AURORA BAY",  name_to_index)

    #CONNECT EVERYTHING TO MENKENT
    add_edge_by_name(G,"MENKENT", "VERNEN WELLS",  name_to_index)

    #CONNECT EVERYTHING TO VERNEN WELLS
    add_edge_by_name(G,"VERNEN WELLS", "TARSH",  name_to_index)
    add_edge_by_name(G,"VERNEN WELLS", "AESIR PASS",  name_to_index)

    #CONNECT EVERYTHING TO TARSH
    add_edge_by_name(G,"TARSH", "MASTIA",  name_to_index)
    add_edge_by_name(G,"TARSH", "CURIA",  name_to_index)

    #CONNECT EVERYTHING TO CURIA
    add_edge_by_name(G,"CURIA", "FENMIRE",  name_to_index)
    add_edge_by_name(G,"CURIA", "BOREA",  name_to_index)
    add_edge_by_name(G,"CURIA", "AESIR PASS",  name_to_index)

    #CONNECT EVERYTHING TO FENMIRE
    add_edge_by_name(G,"FENMIRE", "BARABOS",  name_to_index)

    #CONNECT EVERYTHING TO BARABOS
    add_edge_by_name(G,"BARABOS", "EMERIA",  name_to_index)

    #CONNECT EVERYTHING TO EMERIA
    add_edge_by_name(G,"EMERIA", "BOREA",  name_to_index)

    #CONNECT EVERYTHING TO AESIR PASS
    add_edge_by_name(G,"AESIR PASS", "MARFARK",  name_to_index)

    #CONNECT EVERYTHING TO MARFARK
    add_edge_by_name(G,"MARFARK", "BEKVAM III",  name_to_index)
    add_edge_by_name(G,"MARFARK", "MARTALE",  name_to_index)
    add_edge_by_name(G,"MARFARK", "MATAR BAY",  name_to_index)

    #CONNECT EVERYTHING TO MATAR BAY
    add_edge_by_name(G,"MATAR BAY", "MARTALE",  name_to_index)
    add_edge_by_name(G,"MATAR BAY", "MEISSA",  name_to_index)

    #CONNECT EVERYTHING TO MEISSA
    add_edge_by_name(G,"MEISSA", "WASAT",  name_to_index)
    add_edge_by_name(G,"MEISSA", "X-45",  name_to_index)

    #CONNECT EVERYTHING TO X-45
    add_edge_by_name(G,"X-45", "WASAT",  name_to_index)
    add_edge_by_name(G,"X-45", "VEGA BAY",  name_to_index)

    #CONNECT EVERYTHING TO VEGA BAY
    add_edge_by_name(G,"VEGA BAY", "WASAT",  name_to_index)
    add_edge_by_name(G,"VEGA BAY", "Mox",  name_to_index)
    add_edge_by_name(G,"VEGA BAY", "WEZEN",  name_to_index)

    #CONNECT EVERYTHING TO WEZEN
    add_edge_by_name(G,"WEZEN", "VARYLIA 5",  name_to_index)

    #CONNECT EVERYTHING TO MOX
    add_edge_by_name(G,"Mox", "VARYLIA 5",  name_to_index)

    #CONNECT EVERYTHING TO VARYLIA 5
    add_edge_by_name(G,"VARYLIA 5", "K",  name_to_index)
    add_edge_by_name(G,"VARYLIA 5", "CHOEPESSA IV",  name_to_index)
    add_edge_by_name(G,"VARYLIA 5", "USTOTU",  name_to_index)

    #CONNECT EVERYTHING TO CHOEPESSA IV
    add_edge_by_name(G,"CHOEPESSA IV", "K",  name_to_index)
    add_edge_by_name(G,"CHOEPESSA IV", "USTOTU",  name_to_index)
    add_edge_by_name(G,"CHOEPESSA IV", "FURY",  name_to_index)
    add_edge_by_name(G,"CHOEPESSA IV", "CHARON PRIME",  name_to_index)
    add_edge_by_name(G,"CHOEPESSA IV", "CHARBAL-VII",  name_to_index)

    #CONNECT EVERYTHING TO CHARON PRIME
    add_edge_by_name(G,"CHARON PRIME", "MARTALE",  name_to_index)
    add_edge_by_name(G,"CHARON PRIME", "CHARBAL-VII",  name_to_index)
    add_edge_by_name(G,"CHARON PRIME", "BEKVAM III",  name_to_index)

    #CONNECT EVERYTHING TO CHARBAL-VII
    add_edge_by_name(G,"CHARBAL-VII", "JULHEIM",  name_to_index)
    add_edge_by_name(G,"CHARBAL-VII", "MORT",  name_to_index)

    #CONNECT EVERYTHING TO USTOTU
    add_edge_by_name(G,"USTOTU", "TROOST",  name_to_index)
    add_edge_by_name(G,"USTOTU", "VANDALON IV",  name_to_index)
    add_edge_by_name(G,"USTOTU", "FURY",  name_to_index)

    #CONNECT EVERYTHING TO VANDALON IV
    add_edge_by_name(G,"VANDALON IV", "TROOST",  name_to_index)
    add_edge_by_name(G,"VANDALON IV", "INGMAR",  name_to_index)
    add_edge_by_name(G,"VANDALON IV", "MAIA",  name_to_index)
    add_edge_by_name(G,"VANDALON IV", "MANTES",  name_to_index)

    #CONNECT EVERYTHING TO VINDEMITARIX PRIME
    add_edge_by_name(G,"VINDEMITARIX PRIME", "MEKBUDA",  name_to_index)

    #CONNECT EVERYTHING TO MEKBUD
    add_edge_by_name(G,"MEKBUDA", "CYBERSTAN",  name_to_index)

    #CONNECT EVERYTHING TO CYBERSTAN
    add_edge_by_name(G,"CYBERSTAN", "MERGA IV",  name_to_index)



    #CONNECT EVERYTHING BELOW AUTOMATONS THAT ARE HELD BY HUMANS


    #CONNECT EVERYTHING TO BOREA
    add_edge_by_name(G,"BOREA", "GUNVALD",  name_to_index)
    add_edge_by_name(G,"BOREA", "DUMA TYR",  name_to_index)

    #CONNECT EVERYTHING TO DUMA TYR
    add_edge_by_name(G,"DUMA TYR", "BEKVAM III",  name_to_index)
    add_edge_by_name(G,"DUMA TYR", "OSLO STATION",  name_to_index)
    add_edge_by_name(G,"DUMA TYR", "JULHEIM",  name_to_index)

    #CONNECT EVERYTHING TO BEKVAM III
    add_edge_by_name(G,"BEKVAM III", "JULHEIM",  name_to_index)

    #CONNECT EVERYTHING TO JULHEIM
    add_edge_by_name(G,"JULHEIM", "DOLPH",  name_to_index)

    #CONNECT EVERYTHING TO DOLPH
    add_edge_by_name(G,"DOLPH", "OSLO STATION",  name_to_index)
    add_edge_by_name(G,"DOLPH", "CAPH",  name_to_index)

    #CONNECT EVERYTHING TO OSLO STATION
    add_edge_by_name(G,"OSLO STATION", "GUNVALD",  name_to_index)

    #CONNECT EVERYTHING TO PÖPLI IX
    add_edge_by_name(G,"PÖPLI IX", "MORT",  name_to_index)
    add_edge_by_name(G,"PÖPLI IX", "LASTOFE",  name_to_index)
    add_edge_by_name(G,"PÖPLI IX", "INGMAR",  name_to_index)
    add_edge_by_name(G,"PÖPLI IX", "MANTES",  name_to_index)

    #CONNECT EVERYTHING TO MORT
    add_edge_by_name(G,"MORT", "FURY",  name_to_index)
    add_edge_by_name(G,"MORT", "INGMAR",  name_to_index)

    #CONNECT EVERYTHING TO CAPH
    add_edge_by_name(G,"CAPH", "LASTOFE",  name_to_index)
    add_edge_by_name(G,"CAPH", "CASTOR",  name_to_index)

    #CONNECT EVERYTHING TO CASTOR
    add_edge_by_name(G,"CASTOR", "TIEN KWAN",  name_to_index)

    #CONNECT EVERYTHING TO TIEN KWAN
    add_edge_by_name(G,"TIEN KWAN", "LASTOFE",  name_to_index)
    add_edge_by_name(G,"TIEN KWAN", "DRAUPNIR",  name_to_index)

    #CONNECT EVERYTHING TO DRAUPNIR
    add_edge_by_name(G,"DRAUPNIR", "MANTES",  name_to_index)
    add_edge_by_name(G,"DRAUPNIR", "UBANEA",  name_to_index)
    add_edge_by_name(G,"DRAUPNIR", "MALEVELON CREEK",  name_to_index)

    #CONNECT EVERYTHING TO UBANEA
    add_edge_by_name(G,"UBANEA", "TIBIT",  name_to_index)
    add_edge_by_name(G,"UBANEA", "DURGEN",  name_to_index)

    #CONNECT EVERYTHING TO DURGEN
    add_edge_by_name(G,"DURGEN", "MALEVELON CREEK",  name_to_index)

    #CONNECT EVERYTHING TO MALEVELON CREEK
    add_edge_by_name(G,"MALEVELON CREEK", "MAIA",  name_to_index)
    add_edge_by_name(G,"MALEVELON CREEK", "MANTES",  name_to_index)


    #DO ILLUMINATE SECTORS


    #CONNECT EVERYTHING TO OBARI
    add_edge_by_name(G,"OBARI", "BALDRICK PRIME",  name_to_index)
    add_edge_by_name(G,"OBARI", "VIRIDIA PRIME",  name_to_index)

    #CONNECT EVERYTHING TO BALDRICK PRIME
    add_edge_by_name(G,"BALDRICK PRIME", "ILDUNA PRIME",  name_to_index)
    add_edge_by_name(G,"BALDRICK PRIME", "LIBERTY RIDGE",  name_to_index)

    #CONNECT EVERYTHING TO VIRIDIA PRIME
    add_edge_by_name(G,"VIRIDIA PRIME", "EMORATH",  name_to_index)
    add_edge_by_name(G,"VIRIDIA PRIME", "DILUVIA",  name_to_index)

    #CONNECT EVERYTHING TO EMORATH
    add_edge_by_name(G,"EMORATH", "ILDUNA PRIME",  name_to_index)
    add_edge_by_name(G,"EMORATH", "LIBERTY RIDGE",  name_to_index)
    add_edge_by_name(G,"EMORATH", "EAST IRIDIUM TRADING BAY",  name_to_index)

    #CONNECT EVERYTHING TO LIBERTY RIDGE
    add_edge_by_name(G,"LIBERTY RIDGE", "CANOPUS",  name_to_index)

    #CONNECT EVERYTHING TO CANOPUS
    add_edge_by_name(G,"CANOPUS", "OSUPSAM",  name_to_index)
    add_edge_by_name(G,"CANOPUS", "BUNDA SECUNDUS",  name_to_index)

    #CONNECT EVERYTHING TO BUNDA SECUNDUS
    add_edge_by_name(G,"BUNDA SECUNDUS", "KRAZ",  name_to_index)

    #CONNECT EVERYTHING TO KRAZ
    add_edge_by_name(G,"KRAZ", "LENG SECUNDUS",  name_to_index)

    #CONNECT EVERYTHING TO LENG SCUNDUS
    add_edge_by_name(G,"LENG SECUNDUS", "KLAKA 5",  name_to_index)
    add_edge_by_name(G,"LENG SECUNDUS", "STOUT",  name_to_index)

    #CONNECT EVERYTHING TO STOUT
    add_edge_by_name(G,"STOUT", "STOR THA PRIME",  name_to_index)

    #CONNECT EVERYTHING TO STOR THA PRIME
    add_edge_by_name(G,"STOR THA PRIME", "TERMADON",  name_to_index)
    add_edge_by_name(G,"STOR THA PRIME", "SPHERION",  name_to_index)

    #CONNECT EVERYTHING TO SPHERION
    add_edge_by_name(G,"SPHERION", "SIRIUS",  name_to_index)
    add_edge_by_name(G,"SPHERION", "KNETH PORT",  name_to_index)

    #CONNECT EVERYTHING TO KNETH PORT
    add_edge_by_name(G,"KNETH PORT", "KLAKA 5",  name_to_index)
    add_edge_by_name(G,"KNETH PORT", "BOTEIN",  name_to_index)
    add_edge_by_name(G,"KNETH PORT", "BRINK-2",  name_to_index)

    #CONNECT EVERYTHING TO KLAKA 5
    add_edge_by_name(G, "KLAKA 5", "OSUPSAM",  name_to_index)

    #CONNECT EVERYTHING TO BRINK-2
    add_edge_by_name(G,"BRINK-2", "OSUPSAM",  name_to_index)
    add_edge_by_name(G,"BRINK-2", "EAST IRIDIUM TRADING BAY",  name_to_index)

    #CONNECT EVERYTHING TO EAST IRIDIUM TRADING BAY
    add_edge_by_name(G,"EAST IRIDIUM TRADING BAY", "ELYSIAN MEADOWS",  name_to_index)


    #CONNECT EVERYTHING TO DILUVIA 
    add_edge_by_name(G,"DILUVIA", "SOLGHAST",  name_to_index)
    add_edge_by_name(G,"DILUVIA", "IRULTA",  name_to_index)

    #CONNECT EVERYTHING TO SOLGHAST
    add_edge_by_name(G,"SOLGHAST", "REAF",  name_to_index)
    add_edge_by_name(G,"SOLGHAST", "EFFLUVIA",  name_to_index)

    #CONNECT EVERYTHING TO EFFLUVIA
    add_edge_by_name(G,"EFFLUVIA", "PARSH",  name_to_index)
    add_edge_by_name(G,"EFFLUVIA", "SEYSHEL BEACH",  name_to_index)

    #CONNECT EVERYTHING TO SEYSHEL BEACH
    add_edge_by_name(G,"SEYSHEL BEACH", "FORT SANCTUARY",  name_to_index)
    add_edge_by_name(G,"SEYSHEL BEACH", "KERTH SECUNDUS",  name_to_index)
    add_edge_by_name(G,"SEYSHEL BEACH", "MYRIUM",  name_to_index)

    #CONNECT EVERYTHING TO FORT SANCTUARY
    add_edge_by_name(G,"FORT SANCTUARY", "EUKORIA",  name_to_index)

    #CONNECT EVERTYHING TO IRULTA
    add_edge_by_name(G,"IRULTA", "ELYSIAN MEADOWS",  name_to_index)
    add_edge_by_name(G,"IRULTA", "REAF",  name_to_index)

    #CONNECT EVERYTHING TO ELYSIAN MEADOWS
    add_edge_by_name(G,"ELYSIAN MEADOWS", "CALYPSO",  name_to_index)

    #CONNECT EVERYTHING TO CALYPSO
    add_edge_by_name(G,"CALYPSO", "ALDERIDGE COVE",  name_to_index)
    add_edge_by_name(G,"CALYPSO", "OUTPOST 32",  name_to_index)
    add_edge_by_name(G,"CALYPSO", "ANDAR",  name_to_index)

    #CONNECT EVERYTHING TO ALDERIDGE COVE
    add_edge_by_name(G,"ALDERIDGE COVE", "BELLATRIX",  name_to_index)
    add_edge_by_name(G,"ALDERIDGE COVE", "BOTEIN",  name_to_index)

    #CONNECT EVERYTHING TO BELLATRIX
    add_edge_by_name(G,"BELLATRIX", "KHANDARK",  name_to_index)

    #CONNECT EVERYTHING TO BOTEIN
    add_edge_by_name(G,"BOTEIN", "KHANDARK",  name_to_index)

    #CONNECT EVERYTHING TO KHANDARK
    add_edge_by_name(G,"KHANDARK", "SKAT BAY",  name_to_index)
    add_edge_by_name(G,"KHANDARK", "ASPEROTH PRIME",  name_to_index)

    #CONNECT EVERYTHINGN TO SKAT BAY
    add_edge_by_name(G,"SKAT BAY", "SIRIUS",  name_to_index)
    add_edge_by_name(G,"SKAT BAY", "SIEMNOT",  name_to_index)

    #CONNECT EVERYTHING TO REAF
    add_edge_by_name(G,"REAF", "IRULTA",  name_to_index)
    add_edge_by_name(G,"REAF", "OUTPOST 32",  name_to_index)
    add_edge_by_name(G,"REAF", "PARSH",  name_to_index)

    #CONNECT EVERYTHING TO ANDAR
    add_edge_by_name(G,"ANDAR", "ASPEROTH PRIME",  name_to_index)
    add_edge_by_name(G,"ANDAR", "ALATHFAR XI",  name_to_index)

    #CONNECT EVERYTHING TO ASPEROTH PRIME
    add_edge_by_name(G,"ASPEROTH PRIME", "KEID",  name_to_index)

    #CONNECT EVERYTHING TO KEID
    add_edge_by_name(G,"KEID", "KARLIA",  name_to_index)
    add_edge_by_name(G,"KEID", "SHETE",  name_to_index)

    #CONNECT EVERYTHING TO SHETE
    add_edge_by_name(G,"SHETE", "SETIA",  name_to_index)

    #CONNECT EVERYTHING TO PARSH
    add_edge_by_name(G,"PARSH", "REAF",  name_to_index)
    add_edge_by_name(G,"PARSH", "KERTH SECUNDUS",  name_to_index)
    add_edge_by_name(G,"PARSH", "GENESIS PRIME",  name_to_index)

    #CONNECT EVERYTHING TO GENESIS PRIME
    add_edge_by_name(G,"GENESIS PRIME", "OASIS",  name_to_index)
    add_edge_by_name(G,"GENESIS PRIME", "ALARAPH",  name_to_index)

    #CONNECT EVERYTHING TO ALARAPH
    add_edge_by_name(G,"ALARAPH", "ALATHFAR XI",  name_to_index)
    add_edge_by_name(G,"ALARAPH", "HYDROBIUS",  name_to_index)

    #CONNECT EVERYTHING TO ALATHFAR XI
    add_edge_by_name(G,"ALATHFAR XI", "KARLIA",   name_to_index)
    add_edge_by_name(G,"ALATHFAR XI", "ALARAPH",   name_to_index)

    #CONNECT EVERYTHING TO KARLIA
    add_edge_by_name(G,"KARLIA","KEID",   name_to_index)
    add_edge_by_name(G,"KARLIA","HYDROBIUS",   name_to_index)

    #CONNECT EVERYTHING TO HYDROBIUS
    add_edge_by_name(G,"HYDROBIUS", "HEZE BAY",   name_to_index)
    add_edge_by_name(G,"HYDROBIUS", "SENGE 23",   name_to_index)
    add_edge_by_name(G,"HYDROBIUS", "SETIA",   name_to_index)

    #CONNECT EVERYTHING TO KERTH SECUNDUS
    add_edge_by_name(G,"KERTH SECUNDUS", "GRAFMERE",   name_to_index)
    add_edge_by_name(G,"KERTH SECUNDUS", "MYRIUM",   name_to_index)

    #CONNECT EVERYTHING TO GRAFMERE
    add_edge_by_name(G,"GRAFMERE", "OASIS",   name_to_index)
    add_edge_by_name(G,"GRAFMERE", "IRO",   name_to_index)

    #CONNECT EVERYTHING TO OASIS
    add_edge_by_name(G,"OASIS", "ALAMAK VII",  name_to_index)
    add_edge_by_name(G,"OASIS", "ALARAPH",  name_to_index)

    #CONNECT EVERYTHING TO ALAMAK VII
    add_edge_by_name(G,"ALAMAK VII", "NEW STOCKHOLM",  name_to_index)
    add_edge_by_name(G,"ALAMAK VII", "ALAIRT III",  name_to_index)

    #CONNECT EVERYTHING TO ALAIRT III
    add_edge_by_name(G,"ALAIRT III", "NEW STOCKHOLM", name_to_index)
    add_edge_by_name(G,"ALAIRT III", "HEZE BAY", name_to_index)
    add_edge_by_name(G,"ALAIRT III", "HERTHON SECUNDUS", name_to_index)

    #CONNECT EVERYTHING TO HEZE BAY
    add_edge_by_name(G,"HEZE BAY", "HYDROBIUS", name_to_index)
    add_edge_by_name(G,"HEZE BAY", "RIRGA BAY", name_to_index)

    #CONNECT EVERYTHING TO RIRGA BAY
    add_edge_by_name(G,"RIRGA BAY", "HORT", name_to_index)
    add_edge_by_name(G,"RIRGA BAY", "SEASSE", name_to_index)

    #CONNECT EVERYTHING TO SEASSE
    add_edge_by_name(G,"SEASSE", "SENGE 23", name_to_index)
    add_edge_by_name(G,"SEASSE", "ROGUE 5", name_to_index)

    #CONNECT EVERYTHING TO SENGE 23
    add_edge_by_name(G,"SENGE 23", "SETIA", name_to_index)

    #CONNECT EVERYTHING TO ROGUE 5
    add_edge_by_name(G,"ROGUE 5", "RD-4", name_to_index)

    #CONNECT EVERYTHING TO RD-4
    add_edge_by_name(G,"RD-4", "HESOE PRIME", name_to_index)
    add_edge_by_name(G,"RD-4", "HORT", name_to_index)

    #CONNECT EVERYTHING TO HORT 
    add_edge_by_name(G,"HORT", "HERTHON SECUNDUS", name_to_index)
    add_edge_by_name(G,"HORT", "HESOE PRIME", name_to_index)

    #CONNECT EVERYTHING TO HERTHON SECUNDUS
    add_edge_by_name(G,"HERTHON SECUNDUS", "AIN-5", name_to_index)
    add_edge_by_name(G,"HERTHON SECUNDUS", "ZEA RUGOSIA", name_to_index)

    #CONNECT EVERYTHING TO ZEA RUGOSIA 
    add_edge_by_name(G,"ZEA RUGOSIA", "AFOYAY BAY", name_to_index)
    add_edge_by_name(G,"ZEA RUGOSIA", "HALDUS", name_to_index)

    #CONNECT EVERYTHING TO HALDUS
    add_edge_by_name(G,"HALDUS", "ADHARA", name_to_index)

    #CONNECT EVERYTHING TO ADHARA
    add_edge_by_name(G,"ADHARA", "AFOYAY BAY", name_to_index)
    add_edge_by_name(G,"ADHARA", "MOG", name_to_index)

    #CONNECT EVERYTHING TO AFOYAY BAY
    add_edge_by_name(G,"AFOYAY BAY", "AIN-5", name_to_index)
    add_edge_by_name(G,"AFOYAY BAY", "VALMOX", name_to_index)

    #CONNECT EVERYTHING TO VALMOX
    add_edge_by_name(G,"VALMOX", "IRO", name_to_index)
    add_edge_by_name(G,"VALMOX", "MOG", name_to_index)
    add_edge_by_name(G,"VALMOX", "MYRIUM", name_to_index)

    #CONNECT EVERYTHING TO MOG
    add_edge_by_name(G,"MOG", "REGNUS", name_to_index)

    #CONNECT EVERYTHING TO REGNUS
    add_edge_by_name(G,"REGNUS", "EUKORIA", name_to_index)

    #CONNECT EVERYTHING TO EUKORIA
    add_edge_by_name(G,"EUKORIA", "MYRIUM", name_to_index)

    #CONNECT EVERYTHING TO AIN-5
    add_edge_by_name(G,"AIN-5", "AFOYAY BAY", name_to_index)
    add_edge_by_name(G,"AIN-5", "HERTHON SECUNDUS", name_to_index)
    #CONNECT EVERYTHING TO MYRIUM
    #add_edge_by_name(G,"MYRIUM", "", name_to_index)

    #CONNECT EVERYTHING TO NEW STOCKHOLM
    add_edge_by_name(G,"NEW STOCKHOLM", "IRO", name_to_index)

    #call the function to recompute major order flags
    recompute_major_order_flags(df, G, owner_by_index, major_order_planet_list_index)




    
    df[["enemy_attacking_planet", "enemy_attacking_owner"]] = df["planet_index"].apply(
        lambda x: pd.Series(get_enemy_attacking_planet_and_owner(G, x, owner_by_index))
    )



    # Staging planets OR final target that are already held by Humans
    df["major_order_planet_captured"] = (
        (df["currentOwner"] == "Humans") &
        (df["in_major_order"] == "T")
    ).map({True: "T", False: "F"})


    # Only the actual Major Order objective planet(s)
    df["major_order_completed"] = (
        (df["currentOwner"] == "Humans") &
        (df["planet_index"].isin(major_order_planet_list_index))
    ).map({True: "T", False: "F"})


    #make col for dss_present
    df["DSS_present"] = (df["name"] == dss_planet_orbited).map({True: "T", False: "F"})

    #get date and time
    #df['date'] = 
    df['Y-M-D'] = datetime.now().strftime("%Y-%m-%d")
    df['CST_H_M_S'] = now = datetime.now().strftime("%H:%M:%S")

    df["timestamp"] = datetime.now()

    # Load existing history
    df_history = load_history()

    # Concatenate
    df_history = pd.concat([df_history, df], ignore_index=True)

    # Save back to disk
    save_history(df_history)

    print(f"Saved {len(df)} new rows. Total rows now: {len(df_history)}")
    return df_history



def wait_until_next_hour():
    now = datetime.now()
    next_hour = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
    sleep_seconds = (next_hour - now).total_seconds()
    print(f"Sleeping {int(sleep_seconds)} seconds until {next_hour}")
    time.sleep(sleep_seconds)


while True:
    wait_until_next_hour()
    run_hourly_pipeline()


C:\Users\bulle\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Sleeping 652 seconds until 2026-01-28 14:00:00
[{'id32': 443535211, 'startTime': 61596260, 'progress': [0, 0, 0, 0], 'expiresIn': 454624, 'setting': {'type': 4, 'overrideTitle': 'MAJOR ORDER', 'overrideBrief': 'Hold all designated planets to allow data recovery teams to prove beyond reasonable doubt which of our enemies stole the Star of Peace blueprints.', 'taskDescription': '', 'tasks': [{'type': 13, 'values': [1, 1, 162], 'valueTypes': [3, 11, 12]}, {'type': 13, 'values': [1, 1, 250], 'valueTypes': [3, 11, 12]}, {'type': 13, 'values': [1, 1, 135], 'valueTypes': [3, 11, 12]}, {'type': 13, 'values': [1, 1, 89], 'valueTypes': [3, 11, 12]}], 'rewards': [{'type': 1, 'id32': 897894480, 'amount': 50}], 'reward': {'type': 1, 'id32': 897894480, 'amount': 50}, 'flags': 1}}]
Status: 200
Content-Type: application/json; charset=utf-8
[{"id":443535211,"progress":[0,0,0,0],"title":"MAJOR ORDER","briefing":"Hold all designated planets to allow data recovery teams to prove beyond reasonable doubt wh

In [1]:
# import time
# from datetime import datetime, timedelta

# def wait_until_next_hour():
#     now = datetime.now()
#     next_hour = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
#     sleep_seconds = (next_hour - now).total_seconds()
#     print(f"Sleeping {int(sleep_seconds)} seconds until {next_hour}")
#     time.sleep(sleep_seconds)

# while True:
#     wait_until_next_hour()
#     run_hourly_pipeline()
